# Stage 05 - Readiness Feature Engineering

Build a point-in-time correct feature frame from the same simulated systems and observations used in Stages 01-04.

> The derived label supports analyst prioritization, not an official readiness decision.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_FILE = "readiness_observation_features.csv"
BRONZE_TABLE = "bronze_readiness_observation_features"
DATA_CANDIDATES = [
    Path("/lakehouse/default/Files") / DATA_FILE,
    Path("../data") / DATA_FILE,
    Path("data") / DATA_FILE,
    Path("Files") / DATA_FILE,
]


def locate_data_file(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    checked = ", ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Could not find {DATA_FILE}. Checked: {checked}")


def min_max_scale(series):
    span = series.max() - series.min()
    if span == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.min()) / span

spark_session = globals().get("spark")
if spark_session is not None and spark_session.catalog.tableExists(BRONZE_TABLE):
    frame = spark_session.table(BRONZE_TABLE).toPandas()
    data_source = f"Lakehouse table {BRONZE_TABLE}"
else:
    data_path = locate_data_file(DATA_CANDIDATES)
    frame = pd.read_csv(data_path)
    data_source = str(data_path)

required_columns = {
    "classification", "scenario_id", "simulation_run_id", "test_event_id",
    "track_id", "site_id", "system_instance_id", "system_family",
    "feature_timestamp_utc", "source_snapshot_id", "feature_snapshot_id",
    "baseline_snapshot_id", "finding_snapshot_id", "track_freshness_seconds",
    "gap_count", "quality_rate", "abstract_ack_lag_seconds",
    "health_trend_index", "maintenance_age_days", "maintenance_age_category",
    "test_phase", "baseline_deviation_index", "late_event_count",
    "synthetic_review_priority_label", "synthetic_review_label",
}
missing_columns = sorted(required_columns - set(frame.columns))
assert not missing_columns, f"Readiness input contract violation; missing columns: {missing_columns}"

frame["feature_timestamp_utc"] = pd.to_datetime(frame["feature_timestamp_utc"], utc=True)
frame = frame.sort_values(
    ["scenario_id", "simulation_run_id", "system_instance_id", "feature_timestamp_utc"]
).reset_index(drop=True)
frame["quality_gap"] = 1.0 - frame["quality_rate"]
frame["feature_recency_minutes"] = frame["track_freshness_seconds"] / 60.0
frame["health_decline_flag"] = (frame["health_trend_index"] < 0).astype(int)
frame["maintenance_age_band_index"] = frame["maintenance_age_category"].map(
    {"fresh-service": 0, "steady-cycle": 1, "extended-cycle": 2}
).astype(int)
frame["run_quality_delta"] = frame["quality_rate"] - frame.groupby("simulation_run_id")["quality_rate"].transform("mean")
gap_totals = frame.groupby("simulation_run_id")["gap_count"].transform("sum").replace(0, 1)
frame["run_gap_share"] = frame["gap_count"] / gap_totals
frame["baseline_alignment_gap"] = frame["baseline_deviation_index"] + frame["quality_gap"]
frame["deterministic_baseline_score"] = (
    0.30 * min_max_scale(frame["track_freshness_seconds"])
    + 0.20 * min_max_scale(frame["gap_count"])
    + 0.20 * min_max_scale(frame["quality_gap"])
    + 0.15 * min_max_scale(frame["abstract_ack_lag_seconds"])
    + 0.10 * min_max_scale(frame["baseline_deviation_index"])
    + 0.05 * min_max_scale(frame["maintenance_age_days"])
)

print(f"Loaded {len(frame)} synthetic readiness observations from {data_source}")
frame.head()


In [ ]:
invented_weight_table = pd.DataFrame(
    {
        "feature": [
            "track_freshness_seconds",
            "gap_count",
            "quality_gap",
            "abstract_ack_lag_seconds",
            "baseline_deviation_index",
            "maintenance_age_days",
        ],
        "demo_weight": [0.30, 0.20, 0.20, 0.15, 0.10, 0.05],
    }
)
positive_count = max(1, int(frame["synthetic_review_priority_label"].sum()))
baseline_rank = frame["deterministic_baseline_score"].rank(method="first", ascending=False)
frame["deterministic_baseline_priority_label"] = (baseline_rank <= positive_count).astype(int)

frame[[
    "scenario_id",
    "simulation_run_id",
    "feature_timestamp_utc",
    "source_snapshot_id",
    "feature_snapshot_id",
    "baseline_snapshot_id",
    "finding_snapshot_id",
    "deterministic_baseline_score",
    "deterministic_baseline_priority_label",
]].head()


In [ ]:
feature_columns = ['track_freshness_seconds', 'gap_count', 'quality_rate', 'abstract_ack_lag_seconds', 'health_trend_index', 'maintenance_age_days', 'maintenance_age_category', 'test_phase', 'baseline_deviation_index', 'late_event_count', 'feature_recency_minutes', 'quality_gap', 'health_decline_flag', 'maintenance_age_band_index', 'run_quality_delta', 'run_gap_share', 'baseline_alignment_gap']
lineage_columns = ['classification', 'scenario_id', 'simulation_run_id', 'test_event_id', 'track_id', 'site_id', 'system_instance_id', 'system_family', 'feature_timestamp_utc', 'source_snapshot_id', 'feature_snapshot_id', 'baseline_snapshot_id', 'finding_snapshot_id']
label_columns = [
    "synthetic_review_priority_label",
    "synthetic_review_label",
    "deterministic_baseline_priority_label",
]
name_flags = [
    column
    for column in feature_columns
    if any(token in column for token in ("label", "outcome", "decision", "pass", "fail"))
]
exact_match_flags = [
    column
    for column in feature_columns
    if frame[column].astype(str).equals(frame["synthetic_review_priority_label"].astype(str))
]
assert not set(feature_columns) & set(lineage_columns)
assert not set(feature_columns) & set(label_columns)
assert not name_flags, f"Potential leakage by feature name: {name_flags}"
assert not exact_match_flags, f"Potential leakage by exact feature duplication: {exact_match_flags}"

numeric_correlations = (
    frame[
        [
            "track_freshness_seconds",
            "gap_count",
            "quality_rate",
            "abstract_ack_lag_seconds",
            "health_trend_index",
            "maintenance_age_days",
            "baseline_deviation_index",
            "late_event_count",
            "feature_recency_minutes",
            "quality_gap",
            "health_decline_flag",
            "maintenance_age_band_index",
            "run_quality_delta",
            "run_gap_share",
            "baseline_alignment_gap",
            "synthetic_review_priority_label",
        ]
    ]
    .corr(numeric_only=True)["synthetic_review_priority_label"]
    .drop("synthetic_review_priority_label")
    .abs()
    .sort_values(ascending=False)
    .rename("absolute_correlation")
    .reset_index()
    .rename(columns={"index": "candidate_feature"})
)

numeric_correlations.head(8)


In [ ]:
run_summary = frame.groupby(["scenario_id", "simulation_run_id", "test_phase"], as_index=False).agg(
    observations=("system_instance_id", "count"),
    priority_rate=("synthetic_review_priority_label", "mean"),
    avg_quality_rate=("quality_rate", "mean"),
    avg_baseline_deviation=("baseline_deviation_index", "mean"),
)

spark_session = globals().get("spark")
if spark_session is not None:
    spark_session.createDataFrame(frame).write.mode("overwrite").saveAsTable("silver_readiness_feature_store")
    print("Saved optional Lakehouse table: silver_readiness_feature_store")
else:
    print("Spark session not detected. Skipping optional Lakehouse table write.")

run_summary
